# Notebook de Pruebas: Monitoreo de Calidad del Aire (AQI) y Geocoding

Este notebook permite probar de forma interactiva la conversión de coordenadas a dirección legible (**Reverse Geocoding**) y la consulta del **Índice de Calidad del Aire (AQI)** consumiendo las APIs del ecosistema `aqi-react-app` mediante Programación Orientada a Objetos en Python.

### 1. Importación de Librerías y Definición de Clases POO

In [ ]:
import urllib.request
import urllib.error
import json
from typing import Dict, Any, Tuple

class Location:
    """Clase que representa una ubicación geográfica por sus coordenadas."""
    def __init__(self, latitude: float, longitude: float):
        self.latitude = latitude
        self.longitude = longitude

    @property
    def latitude(self) -> float:
        return self.__latitude

    @latitude.setter
    def latitude(self, value: float):
        if not (-90.0 <= value <= 90.0):
            raise ValueError(f"Latitud fuera de rango (-90 a 90): {value}")
        self.__latitude = value

    @property
    def longitude(self) -> float:
        return self.__longitude

    @longitude.setter
    def longitude(self, value: float):
        if not (-180.0 <= value <= 180.0):
            raise ValueError(f"Longitud fuera de rango (-180 a 180): {value}")
        self.__longitude = value

    def to_tuple(self) -> Tuple[float, float]:
        return (self.__latitude, self.__longitude)

    def __str__(self) -> str:
        return f"({self.__latitude:.4f}, {self.__longitude:.4f})"

print("[OK] Clase Location cargada exitosamente.")

### 2. Definición de Servicios de API (Reverse Geocode & Air Quality)

In [ ]:
class ReverseGeocodeService:
    def __init__(self, api_key: str):
        self.__api_key = api_key
        self.__endpoint = "https://api.geoapify.com/v1/geocode/reverse"

    def get_readable_location(self, location: Location) -> str:
        lat, lon = location.to_tuple()
        url = f"{self.__endpoint}?lat={lat}&lon={lon}&apiKey={self.__api_key}"
        req = urllib.request.Request(url, headers={'User-Agent': 'PythonAQIMonitor/1.0'})
        with urllib.request.urlopen(req) as response:
            data = json.loads(response.read().decode('utf-8'))
            features = data.get("features", [])
            if features:
                return features[0].get("properties", {}).get("formatted", "Ubicación desconocida")
            return "Ubicación no encontrada"

class AirQualityService:
    AQI_DESCRIPTIONS = {
        1: ("Excelente / Bueno", "Aire limpio, mínimo riesgo."),
        2: ("Aceptable / Moderado", "Calidad de aire aceptable."),
        3: ("Moderado / Sensible", "Grupos sensibles pueden experimentar molestias."),
        4: ("Malo / Dañino", "Dañino para la salud."),
        5: ("Muy Malo / Peligroso", "Alerta de salud grave.")
    }

    def __init__(self, api_key: str):
        self.__api_key = api_key
        self.__endpoint = "http://api.openweathermap.org/data/2.5/air_pollution"

    def fetch_air_quality(self, location: Location) -> Dict[str, Any]:
        lat, lon = location.to_tuple()
        url = f"{self.__endpoint}?lat={lat}&lon={lon}&appid={self.__api_key}"
        req = urllib.request.Request(url, headers={'User-Agent': 'PythonAQIMonitor/1.0'})
        with urllib.request.urlopen(req) as response:
            return json.loads(response.read().decode('utf-8'))

print("[OK] Servicios de API cargados.")

### 3. Prueba Interactiva de Consulta

In [ ]:
# Coordenadas de prueba (Ejemplo: Asunción, Paraguay)
LAT = -25.2867
LON = -57.6470

GEO_KEY = 'e94fd042131f45f18a7a4c89d5b8276d'
AQI_KEY = '2f9f437fc127edba8c7068fe3bd209f4'

loc = Location(LAT, LON)
geo_service = ReverseGeocodeService(GEO_KEY)
aqi_service = AirQualityService(AQI_KEY)

direccion = geo_service.get_readable_location(loc)
raw_data = aqi_service.fetch_air_quality(loc)
aqi_val = raw_data['list'][0]['main']['aqi']
contaminantes = raw_data['list'][0]['components']

print(f"Coordenadas: {loc}")
print(f"Ubicación Legible: {direccion}")
print(f"Nivel AQI: {aqi_val}/5 -> {AirQualityService.AQI_DESCRIPTIONS[aqi_val][0]}")
print("Contaminantes:", contaminantes)